In [ ]:
import os, sys
user_home = os.environ['HOME']
repo_dir = os.path.join(user_home, 'Documents/git/stoic2021')
sys.path.append(repo_dir)
# print(user_home)
print(sys.path)

from mmdet.datasets.transform4med.load_dicom import *
from mmdet.datasets.transform4med.io4med import IO4Nii
from scripts.visual_func import plot2Image, plotNImage, save_fig

from scipy import ndimage
from skimage.morphology.selem import ball, disk
from skimage.morphology import binary_opening, remove_small_holes
from skimage.measure import label, regionprops
from skimage.filters import threshold_otsu
import cv2



In [ ]:
import cc3d 
np.set_printoptions(precision=2, suppress=True)

def get_legs_2d(image_2d, value_range = (-900, 3000), 
                bone_thresh = None, 
                pivot_axis = 1, 
                small_region = 200, 
                pid = None,
                verb = False):

    image_2d = np.clip(image_2d, value_range[0], value_range[1])
    if bone_thresh is None:
        bone_thresh = threshold_otsu(image_2d)

    mask_bi = image_2d > bone_thresh
    mask_open = binary_opening(mask_bi, disk(5)) 

    labels_out, N = cc3d.connected_components(mask_open, return_N=True, connectivity=8) # free
    # print('num region', N)
    if N <= 1:
        return mask_open, labels_out, labels_out
    # Image statistics like voxel counts, bounding boxes, and centroids.
    stats = cc3d.statistics(labels_out)
    region_centroids = np.array(stats['centroids']) # nx3
    region_counts = np.array(stats['voxel_counts'], dtype = np.int64) # nx1
    if verb: print(region_centroids)
    if verb: print(region_counts)

    r2r_center = np.abs(region_centroids[:, None, :] - region_centroids[None, ...])[..., pivot_axis] # nxnx3
    r2r_count = np.abs(region_counts[:, None] - region_counts[None, :])
    r2c_img = np.abs(region_centroids[:, pivot_axis] - image_2d.shape[pivot_axis]//2)
    
    r2imgc_sort_index = np.argsort(r2c_img)
    r2imgc_close2 = r2imgc_sort_index[1:3]
    r2img_remote = r2imgc_sort_index[3:]
    if verb: print('R2C', r2c_img, r2imgc_close2)
    if r2imgc_close2[0] > r2imgc_close2[1]:
        r2imgc_close2[0], r2imgc_close2[1] = r2imgc_close2[1], r2imgc_close2[0]

    if verb: print(f'r2r center { r2r_center.shape} \n', r2r_center.round(1))
    if verb: print(f'r2r count {r2r_count.shape} {r2r_count.dtype} \n', r2r_count.round(1))

    center_dist_index = (1, 2)
    center_dist_min = r2r_center[1, 2]

    count_diff_index = (1, 2)
    count_diff_min = r2r_count[1, 2]

    for i in range(1, N + 1):
        # skip region that has an area smaller than 25
        if region_counts[i] < small_region or i in r2img_remote: continue
        
        for j in range(i + 1, N + 1):
            if region_counts[j] < small_region or j in r2img_remote: continue
            # print(f'{i} {j}')
            if r2r_center[i, j] < center_dist_min:
                center_dist_min = r2r_center[i, j]
                center_dist_index = (i, j)
            
            if r2r_count[i, j] < count_diff_min:
                count_diff_min = r2r_count[i, j]
                count_diff_index = (i, j)
    
    if verb: print(f'CenterDistIndex {center_dist_index} CountDiffDist{count_diff_index}')
    if verb: print(f'CenterDistMin{center_dist_min:2f} CountDiffMin{count_diff_min}')
    target_region_index = count_diff_index

    if count_diff_index != center_dist_index:
        raise ValueError(f'[PID{pid}] CenterDistIndex {center_dist_index} not equal to CountDiffDist{count_diff_index} ')

    if target_region_index != tuple(r2imgc_close2):
        raise ValueError(f'[PID{pid}] CountDiffDist {center_dist_index} not equal to Region2ImgCenterDist{r2imgc_close2}')
        target_region_index = r2imgc_close2
    

    leg_mask = np.max(np.stack([labels_out == i for i in target_region_index], axis = 0), axis = 0)
    
    # rst_array = cv2.morphologyEx(input_array, cv2.MORPH_DILATE, element)

    return mask_open, labels_out, leg_mask

In [ ]:
data_dir = Path('/Users/monolith/Desktop/leg_seg/leg_nii')
# test_pid = 'case_leg_dicom_PET084053-NAF_1.2.156.112605.189250940724405.210325230838.3.5184.916816.nii.gz'
# test_pid = 
test_pids = [a for a in os.listdir(data_dir) if 'nii' in a]

for test_pid in test_pids:
# test_pid = 'case_leg_dicom_ZHANG^YUE_CT22004812_1.2.840.113619.2.278.3.2831163265.332.1642986487.727.nii.gz'
    pid_fp = data_dir/test_pid
    image_3d, af_mat = IO4Nii.read(pid_fp, verbose=False)
    num_slices = image_3d.shape[-1]
    for i in range(0, num_slices, 5):
        image_slice = image_3d[..., i]
        image_mask, labels_out, leg_mask = get_legs_2d(image_slice)
        
        pid_dir = data_dir/test_pid.split('.nii')[0]
        pid_dir.mkdir(parents=True, exist_ok=True)
        # fig = plot2Image(image_slice, image_mask, cmap = 'gray',
        #                 save_dir= pid_dir,  
        #                 fig_title=f'leg_slice_{i}')
        fig = plotNImage([image_slice, image_mask, labels_out, leg_mask], 
                        cmap = 'gray', is_close=True, fig_title=f'leg_slice_{i}')
        save_fig(fig, pid_dir/f'leg_slice_{i}')

# fig = plotNImage([image_3d[..., a] for a in range(0, 200, 20)], rows = 2, cmap='gray')

In [ ]:

slice_ix = 70
image_slice = image_3d[..., slice_ix]
image_mask, labels_out, leg_mask = get_legs_2d(image_slice, verb = True)

aa = plotNImage([image_slice, image_mask, labels_out, leg_mask])